# NLLB-200 training: load data -> train all configurations -> evaluate -> demo

This notebook is intentionally thin. All the logic that matters - LoRA wrapping, the
verified embedding-unfreezing fix, data loading, metrics, training - lives in the shared
`.py` modules alongside this notebook (`arch_config.py`, `data_io.py`, `eda.py`,
`peft_setup.py`, `metrics.py`, `train.py`, `inference.py`). `mt5_training.ipynb` imports
the exact same modules, so both notebooks share one implementation rather than two
copies that could drift apart.

**Data note:** this notebook loads the already-split, already-built curriculum files -
it does not merge or re-split anything. See `data_io.py`'s docstring for why.

**Pipeline position:** `00_data_prep.ipynb` builds `data/`, `01_eda.ipynb` reports on it in
depth - this is `02_nllb_training.ipynb`, the third notebook, and doesn't repeat either of
those. What's new here for training/eval: the test partition is now three independent sets
- `test_psa`, `test_bible`, `test_general` - instead of one PSA-only `test.csv`, so Section 7
scores every configuration against all three separately (chrF2++ per test set, per
direction), not one pooled number.

**Where this runs.** This project's layout is `Final_Training/{arch_config.py, data_io.py,
eda.py, peft_setup.py, metrics.py, train.py, inference.py, experiment_log.csv, raw_data/,
data/, Notebooks/}` - this notebook lives in `Notebooks/`, `find_project_root()` below walks
up to `Final_Training/` (where the shared `.py` modules and `experiment_log.csv` live), and
every path (`data/`, `checkpoints/`, `logs/`, `deployment/`) resolves relative to that root,
never to the separate, independent project this was adapted from.

## 1. Setup

In [1]:
!pip install -q torch transformers accelerate sentencepiece evaluate sacrebleu mlflow peft matplotlib pandas datasets


### GPU / CUDA check — run this before anything else

Confirms `torch` can actually see the GPU before spending any time on data loading or
training. If `torch.cuda.is_available()` prints `False` here, stop and fix that first —
everything downstream will silently fall back to CPU otherwise, which won't error out,
it will just be catastrophically slow and easy to mistake for a hung cell.

In [2]:
import torch
print(f"torch version       : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (torch): {torch.version.cuda}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"total GPU memory    : {total_gb:.1f} GB")
else:
    print("\nWARNING: no GPU visible to torch. If nvidia-smi shows a GPU but this says False,")
    print("the installed torch build likely doesn't match this environment's CUDA driver.")
    print("Fix with something like:")
    print("  pip install torch --index-url https://download.pytorch.org/whl/cu121")
    print("(cu121 is broadly compatible with CUDA 12.x drivers; check pytorch.org for the")
    print(" exact build matching this specific environment if that doesn't resolve it)")


torch version       : 2.11.0+cu128
CUDA available      : True
GPU                 : NVIDIA H100 PCIe
CUDA version (torch): 12.8
total GPU memory    : 79.2 GB


In [3]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT} - working directory changed here so all relative paths resolve correctly")

from arch_config import ARCH_CONFIG, MAX_LEN, LORA_KWARGS, SEED, DATA_DIR, CHECKPOINTS_DIR, LOGS_DIR, DEPLOYMENT_DIR, MLFLOW_DB
import data_io, eda, peft_setup, metrics, inference
from train import train_stage

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from datasets import Dataset

set_seed(SEED)
ARCHITECTURE = "nllb"
cfg = ARCH_CONFIG[ARCHITECTURE]
print(f"architecture: {ARCHITECTURE}")
print(f"checkpoint  : {cfg['checkpoint']}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

# Loud, impossible-to-miss check before anything expensive runs: confirm every
# output path this notebook writes to actually resolves under THIS project root,
# not the separate, independent project this was adapted from. Cheap to print,
# expensive to discover wrong after 8 training runs and a deployment step.
# mlflow db uses a sqlite file, not a directory - the installed MLflow blocks the
# plain filesystem backend by default (see train.py's _ensure_isolated_mlflow_tracking
# docstring); this path is just where the .db file itself will be created.
print("\n" + "=" * 70)
print("OUTPUT PATHS FOR THIS RUN - verify these before running Section 6")
print("=" * 70)
print(f"  data        -> {(PROJECT_ROOT / DATA_DIR).resolve()}")
print(f"  checkpoints -> {(PROJECT_ROOT / CHECKPOINTS_DIR).resolve()}")
print(f"  logs        -> {(PROJECT_ROOT / LOGS_DIR).resolve()}")
print(f"  deployment  -> {(PROJECT_ROOT / DEPLOYMENT_DIR).resolve()}")
print(f"  mlflow db   -> {(PROJECT_ROOT / MLFLOW_DB).resolve()}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - if this project")
    print("  lives in a Final_Training/ folder, this almost certainly means arch_config.py (and")
    print("  the other shared .py modules) haven't been placed there yet, and find_project_root()")
    print("  walked up past it to an older project's copy instead. Fix that before running")
    print("  Section 6 - every checkpoint/log/deployment/mlruns path above would otherwise land")
    print("  in the wrong project.")
print("=" * 70)


project root: /home/jovyan/PSA/Final_Training - working directory changed here so all relative paths resolve correctly
architecture: nllb
checkpoint  : facebook/nllb-200-distilled-600M
GPU         : NVIDIA H100 PCIe

OUTPUT PATHS FOR THIS RUN - verify these before running Section 6
  data        -> /home/jovyan/PSA/Final_Training/data
  checkpoints -> /home/jovyan/PSA/Final_Training/checkpoints
  logs        -> /home/jovyan/PSA/Final_Training/logs
  deployment  -> /home/jovyan/PSA/Final_Training/deployment
  mlflow db   -> /home/jovyan/PSA/Final_Training/mlflow.db


## 2. Data overview

Full EDA (domain composition, length distributions, vocabulary size, an out-of-vocabulary
analysis per test set) now lives in `01_eda.ipynb`, run separately before this notebook -
reproducing it here would just be a second copy that could drift from the first. This cell
is only a quick sanity check that the expected files are present and the row counts match
what `01_eda.ipynb`/`mixture.json` last reported.

In [4]:
flat = data_io.load_flat_splits(DATA_DIR)
for name, d in flat.items():
    print(f"{name:20s} {len(d):>8,} rows")

train                  70,480 rows
validation_general      6,080 rows
validation_psa          1,751 rows
test                    9,142 rows


## 3. Load the already-split curriculum data

In [5]:
tokenizer = AutoTokenizer.from_pretrained(cfg["checkpoint"])
tokenizer = peft_setup.extend_tokenizer_for_new_language(tokenizer, cfg)
print(f"vocab size: {len(tokenizer):,}")
print(f"new token {cfg['tgt_code']!r} will be seeded from {cfg['related_code']!r} "
      f"(peft_setup.resize_model_for_new_language does the actual copy, per-run below)")

Added new token 'guz_Latn' at id 256204
vocab size: 256,205
new token 'guz_Latn' will be seeded from 'lug_Latn' (peft_setup.resize_model_for_new_language does the actual copy, per-run below)


In [6]:
curriculum = data_io.load_curriculum(DATA_DIR)
psa_only_rows = data_io.derive_psa_only(curriculum["stage2"])
eda.curriculum_size_report(curriculum, psa_only_rows=psa_only_rows)

# The three-part test set: data_io.load_curriculum() only knows the single combined
# "test" name (that contract predates the three-way split) - load the three individually
# here via the same public load_jsonl() helper, no changes to data_io.py needed.
test_sets = {name: data_io.load_jsonl(Path(DATA_DIR) / f"{name}.jsonl")
             for name in ["test_psa", "test_bible", "test_general"]}
for name, rows in test_sets.items():
    print(f"  {name:20s} {len(rows):>7,} examples")

derive_psa_only: kept 123,568 psa-tagged rows, dropped 41,189 replay-tagged rows (25% of stage2)
  cross-check: this count should exactly match the 'psa_up' figure printed when stage2.jsonl was originally built - if it doesn't, the file has likely changed since then.
=== Curriculum sizes (direction-expanded) ===
  stage1                104,234 examples
  stage2                164,757 examples
  mixed                 227,802 examples
  validation_general      7,851 examples
  validation_psa          3,405 examples
  test                   13,629 examples
  psa_only              123,568 examples  (derived from stage2, replay rows excluded - no new file)
  test_psa               3,020 examples
  test_bible             2,576 examples
  test_general           8,033 examples


## 4. Tokenizer: add the real `guz_Latn` token

Not a placeholder reusing Swahili's tag - a genuine new token, its embedding initialized
by copying whichever language `arch_config.py`'s `related_code` names (see the printed
line below for which one this run actually used - check the seed-language discussion
elsewhere in this project before assuming that's still Kikuyu; it may have changed).

## 5. Tokenization

In [7]:
def nllb_preprocess(batch):
    all_ids = []
    for sl, src, tgt in zip(batch["src_lang"], batch["src"], batch["tgt"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = cfg["tgt_code"]
        enc = tokenizer(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
    return {"input_ids": [e["input_ids"] for e in all_ids],
            "attention_mask": [e["attention_mask"] for e in all_ids],
            "labels": [e["labels"] for e in all_ids]}

def tokenize_rows(rows):
    ds = Dataset.from_list(rows)
    return ds.map(nllb_preprocess, batched=True, batch_size=16)

tok = {
    "stage1": tokenize_rows(curriculum["stage1"]),
    "stage2": tokenize_rows(curriculum["stage2"]),
    "mixed": tokenize_rows(curriculum["mixed"]),
    "psa_only": tokenize_rows(psa_only_rows),
    "validation_general": tokenize_rows(curriculum["validation_general"]),
    "validation_psa": tokenize_rows(curriculum["validation_psa"]),
    "test": tokenize_rows(curriculum["test"]),
}
print("tokenization complete")


Map:   0%|          | 0/104234 [00:00<?, ? examples/s]

Map:   0%|          | 0/164757 [00:00<?, ? examples/s]

Map:   0%|          | 0/227802 [00:00<?, ? examples/s]

Map:   0%|          | 0/123568 [00:00<?, ? examples/s]

Map:   0%|          | 0/7851 [00:00<?, ? examples/s]

Map:   0%|          | 0/3405 [00:00<?, ? examples/s]

Map:   0%|          | 0/13629 [00:00<?, ? examples/s]

tokenization complete


## 6. Run all NLLB configurations, one cell per run

Every one of the 8 fine-tuned configurations gets its own cell — each prints its own
`run_id` as the first line of output (from `train_stage()`), and the markdown label above
each cell states which run it is, so a failure or an odd result is traceable to one specific
cell, not buried inside a loop that ran several configurations together. See
`experiment_log.csv` for the full planned set (9 rows: zero-shot + these 8).

**Run: `nllb_stage1_lora_emb`** and **`nllb_stage1_lora_only`** — both from the base
checkpoint, both on Stage 1 (general Ekegusii) data; the `_lora_only`/`_lora_emb` pair
completes the same ablation Stage 2/Mixed/PSA-only already each get (see the next markdown
cell for what that pair actually means in this codebase). Every *other* NLLB run either
branches from `nllb_stage1_lora_emb`'s output (Stage 2) or starts independently from the
base checkpoint (Mixed, PSA-only) - `nllb_stage1_lora_only` is a leaf like Mixed/PSA-only,
nothing branches from it.

In [8]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(cfg["checkpoint"])
base_model = peft_setup.resize_model_for_new_language(base_model, tokenizer, cfg)
zeroshot_path = f"{CHECKPOINTS_DIR}/nllb_zeroshot"
base_model.save_pretrained(zeroshot_path, safe_serialization=True)
tokenizer.save_pretrained(zeroshot_path)
del base_model
torch.cuda.empty_cache()
print(f"zero-shot baseline saved: {zeroshot_path}")


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

zero-shot baseline saved: checkpoints/nllb_zeroshot


In [10]:
_, stage1_path, _ = train_stage(
    "nllb_stage1_lora_emb", ARCHITECTURE, cfg, "stage1", cfg["checkpoint"],
    tok["stage1"], tok["validation_general"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs = {"stage1_lora_emb": stage1_path}


  nllb_stage1_lora_emb   arch=nllb  stage=stage1  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_stage1_lora_emb
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'
trainable params: 264,713,216 || all params: 617,432,064 || trainable%: 42.8733


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,2.656691,2.506409,6.431404,31.511618
2,2.344576,2.159529,8.524337,35.178443
3,2.166924,1.990497,10.260123,37.861654
4,2.003420,1.891720,11.495631,39.352444
5,1.935967,1.829429,12.327050,40.377690
6,1.919754,1.789634,12.717776,40.882955
7,1.915941,1.768335,13.055632,41.339605
8,1.896484,1.760766,13.121874,41.444602


/opt/conda/lib/python3.11/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
2026/09/05 04:56:41 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(sqlite3.OperationalError) database is locked
[SQL: SELECT runs.run_uuid AS runs_run_uuid, runs.name AS runs_name, runs.source_type AS runs_source_type, runs.source_name AS runs_source_name, runs.entry_point_name AS runs_entry_point_name, runs.user_id AS runs_user_id, runs.status AS runs_status, runs.start_time AS runs_start_time, runs.end_time AS runs_end_time, runs.deleted_time AS runs_deleted_time, runs.source_version AS runs_source_version, runs.lifecycle_stage AS runs_lifecycle_stage, runs.artifact_uri AS runs_artifact_uri, runs.experiment_id AS runs_experiment_id 
FROM runs 
WHERE runs.run_uuid = ?]
[parameters: ('fe8497fbdcf04110b53ba63f83bf59f4',)]
(Background on this 

nllb_stage1_lora_emb training time: 189.2 minutes


/opt/conda/lib/python3.11/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Note: the embedding update lives outside peft's tracking (see peft_setup.py), so this adapter-only save does NOT include it. The merged checkpoint below is the complete, deployment-ready artifact for this run.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_stage1_lora_only`** — same data and init as `nllb_stage1_lora_emb` above,
the other half of that ablation pair. Read literally, `use_lora=False` below means this run
skips LoRA entirely and falls back to partial layer-freezing (`peft_setup.freeze_encoder_layers`)
instead - `wrap_for_training()` has no way to request "LoRA active, embedding still frozen"
as a third option, since embedding-unfreezing is gated on `cfg["unfreeze_embeddings"]`
(a fixed, architecture-level setting) whenever `use_lora=True`. So the actual comparison
every `_lora_only` vs. `_lora_emb` pair in this notebook makes - here and for Stage 2/Mixed/
PSA-only alike - is "layer-freezing baseline" vs. "LoRA + embedding unfreeze together", not
"LoRA alone" vs. "LoRA + embedding". Kept consistent with how the other three pairs already
ran rather than changed quietly for just this one - flag it if a true three-way split
(no-LoRA / LoRA-only / LoRA+embedding) is what you actually want compared, since that needs
`wrap_for_training()` changed, not just this cell.

In [ ]:
_, stage1_lora_only_path, _ = train_stage(
    "nllb_stage1_lora_only", ARCHITECTURE, cfg, "stage1", cfg["checkpoint"],
    tok["stage1"], tok["validation_general"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["stage1_lora_only"] = stage1_lora_only_path

  nllb_stage1_lora_only   arch=nllb  stage=stage1  init=facebook/nllb-200-distilled-600M  use_lora=False
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_stage1_lora_only
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,1.413581,1.384328,20.000254,48.044910
2,1.203708,1.208121,24.093016,51.349848
3,1.080132,1.120766,25.971199,52.990262
4,0.923313,1.069819,27.406947,53.977539
5,0.867352,1.038570,28.483166,54.720690
6,0.820817,1.016459,29.237961,55.303230
7,0.791300,1.007419,29.686728,55.598622


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_stage2_lora_only`** — branches from the Stage 1 checkpoint above; embedding stays frozen from here on.

In [ ]:
_, stage2_lora_only_path, _ = train_stage(
    "nllb_stage2_lora_only", ARCHITECTURE, cfg, "stage2", str(stage1_path),
    tok["stage2"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["stage2_lora_only"] = stage2_lora_only_path


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


nllb_stage2_lora_only training time: 154.6 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_stage2_lora_emb`** — same branch point as above; embedding keeps updating through specialization.

In [ ]:
import mlflow

while mlflow.active_run() is not None:
    print(f"ending stale run: {mlflow.active_run().info.run_id}")
    mlflow.end_run()

print("clean - no active runs remaining")

clean - no active runs remaining


In [ ]:
_, stage2_lora_emb_path, _ = train_stage(
    "nllb_stage2_lora_emb", ARCHITECTURE, cfg, "stage2", str(stage1_path),
    tok["stage2"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["stage2_lora_emb"] = stage2_lora_emb_path


  nllb_stage2_lora_emb   arch=nllb  stage=stage2  init=checkpoints/nllb_stage1_lora_emb/merged  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_stage2_lora_emb
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

embedding matrix already sized for the extended vocab (256205) - assuming 'guz_Latn' is already initialized from a prior stage, not re-copying
trainable params: 264,713,216 || all params: 617,432,064 || trainable%: 42.8733


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,2.299184,2.281837,9.606128,32.897315
2,2.069686,2.038776,11.605960,36.001207
3,1.874459,1.913931,13.035297,37.733636
4,1.910212,1.834517,13.504627,38.267851
5,1.807547,1.789162,14.679038,39.483586
6,1.691088,1.759352,15.221409,40.052390
7,1.753607,1.740272,15.315236,40.170702
8,1.742817,1.734113,15.226711,40.106719


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



nllb_stage2_lora_emb training time: 139.3 minutes
Note: the embedding update lives outside peft's tracking (see peft_setup.py), so this adapter-only save does NOT include it. The merged checkpoint below is the complete, deployment-ready artifact for this run.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_mixed_lora_only`** — fresh from the base checkpoint, not from Stage 1 (this is the single-pass control).

In [18]:
import logging
import warnings

logging.getLogger("mlflow").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

In [19]:
_, mixed_lora_only_path, _ = train_stage(
    "nllb_mixed_lora_only", ARCHITECTURE, cfg, "mixed", cfg["checkpoint"],
    tok["mixed"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["mixed_lora_only"] = mixed_lora_only_path


  nllb_mixed_lora_only   arch=nllb  stage=mixed  init=facebook/nllb-200-distilled-600M  use_lora=False
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_mixed_lora_only
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,1.392494,1.423488,21.359038,45.647877
2,1.031896,1.282187,24.101933,48.165835
3,0.869723,1.250932,27.682323,50.332764
4,0.745661,1.257429,31.052774,52.740822
5,0.658051,1.282511,32.744438,54.090403
6,0.604025,1.299341,35.126835,55.458100
7,0.566203,1.320129,35.849788,55.989943
8,0.531121,1.322354,36.428559,56.436833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


nllb_mixed_lora_only training time: 180.8 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_mixed_lora_emb`**

In [20]:
_, mixed_lora_emb_path, _ = train_stage(
    "nllb_mixed_lora_emb", ARCHITECTURE, cfg, "mixed", cfg["checkpoint"],
    tok["mixed"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["mixed_lora_emb"] = mixed_lora_emb_path


  nllb_mixed_lora_emb   arch=nllb  stage=mixed  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_mixed_lora_emb
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'
trainable params: 264,713,216 || all params: 617,432,064 || trainable%: 42.8733


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,2.571840,2.395759,9.069244,31.755832
2,2.237364,2.110819,10.445112,34.089791
3,2.050623,1.971868,12.009091,36.587767
4,1.944326,1.886495,13.414664,38.134255
5,1.903506,1.830302,13.897432,38.710383
6,1.844598,1.793974,14.724355,39.637843
7,1.824335,1.775188,15.158591,40.030201
8,1.766145,1.769617,15.064564,40.030932


nllb_mixed_lora_emb training time: 183.3 minutes
Note: the embedding update lives outside peft's tracking (see peft_setup.py), so this adapter-only save does NOT include it. The merged checkpoint below is the complete, deployment-ready artifact for this run.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_psaonly_lora_only`** — fresh from the base checkpoint, PSA data only, no general data anywhere.

In [21]:
_, psaonly_lora_only_path, _ = train_stage(
    "nllb_psaonly_lora_only", ARCHITECTURE, cfg, "psa_only", cfg["checkpoint"],
    tok["psa_only"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["psaonly_lora_only"] = psaonly_lora_only_path


  nllb_psaonly_lora_only   arch=nllb  stage=psa_only  init=facebook/nllb-200-distilled-600M  use_lora=False
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_psaonly_lora_only
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,1.310400,1.443919,21.211715,45.489242
2,0.867706,1.304139,25.094895,48.669871
3,0.682497,1.281444,28.933456,51.172828
4,0.566316,1.299360,32.216117,53.170342
5,0.390407,1.325132,34.294027,54.837337
6,0.367933,1.345269,36.015835,55.986433
7,0.324362,1.365302,37.203060,56.432222
8,0.306686,1.370862,37.652361,56.930890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


nllb_psaonly_lora_only training time: 110.5 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Run: `nllb_psaonly_lora_emb`**

In [ ]:
_, psaonly_lora_emb_path, _ = train_stage(
    "nllb_psaonly_lora_emb", ARCHITECTURE, cfg, "psa_only", cfg["checkpoint"],
    tok["psa_only"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
runs["psaonly_lora_emb"] = psaonly_lora_emb_path

print("\nall 8 fine-tuned NLLB runs complete:")
for name, path in runs.items():
    print(f"  {name:20s} -> {path}")


  nllb_psaonly_lora_emb   arch=nllb  stage=psa_only  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_psaonly_lora_emb
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

resized embeddings 256206 -> 256205 and initialized 'guz_Latn' from 'lug_Latn'
trainable params: 264,713,216 || all params: 617,432,064 || trainable%: 42.8733


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,2.624098,2.449255,7.808627,29.768860
2,2.224312,2.164070,9.673694,33.104310
3,2.059373,2.020874,11.208923,35.264688
4,2.014315,1.932330,12.095961,36.475481
5,1.843430,1.879327,12.977029,37.875625
6,1.888028,1.841583,13.591172,38.291164
7,1.868805,1.823320,13.911020,38.687219
8,1.878538,1.817022,13.846675,38.766980


nllb_psaonly_lora_emb training time: 115.1 minutes
Note: the embedding update lives outside peft's tracking (see peft_setup.py), so this adapter-only save does NOT include it. The merged checkpoint below is the complete, deployment-ready artifact for this run.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


all 8 fine-tuned NLLB runs complete:
  stage1_lora_emb      -> checkpoints/nllb_stage1_lora_emb/merged
  stage1_lora_only     -> checkpoints/nllb_stage1_lora_only/merged
  stage2_lora_only     -> checkpoints/nllb_stage2_lora_only/merged
  stage2_lora_emb      -> checkpoints/nllb_stage2_lora_emb/merged
  mixed_lora_only      -> checkpoints/nllb_mixed_lora_only/merged
  mixed_lora_emb       -> checkpoints/nllb_mixed_lora_emb/merged
  psaonly_lora_only    -> checkpoints/nllb_psaonly_lora_only/merged
  psaonly_lora_emb     -> checkpoints/nllb_psaonly_lora_emb/merged


## 7. Evaluate every configuration on all three test sets

Every one of the 8 checkpoints (zero-shot + 7 fine-tuned) is scored against `test_psa`,
`test_bible`, and `test_general` independently, in both directions (English->Ekegusii,
Kiswahili->Ekegusii) - not one pooled PSA-only number like before. chrF2++ is the metric
that drives any model-selection decision (see `metrics.py`'s docstring for why, not BLEU);
BLEU is still reported alongside it for readers used to seeing it.

In [ ]:
test_specs = {"bible": test_sets["test_bible"], "psa": test_sets["test_psa"],
              "general": test_sets["test_general"]}
directions = [("en-guz", "eng_Latn", "en"), ("sw-guz", "swh_Latn", "sw")]

all_paths = {"zeroshot": zeroshot_path, **{k: str(v) for k, v in runs.items()}}
results = []

for run_name, path in all_paths.items():
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tok_for_run = AutoTokenizer.from_pretrained(path)

    for test_name, rows in test_specs.items():
        for direction, src_lang_code, lang_arg in directions:
            subset = [r for r in rows if r["src_lang"] == src_lang_code]
            if not subset:
                continue
            preds = [inference.generate_one(model, tok_for_run, r["src"], lang_arg, cfg,
                                             ARCHITECTURE, MAX_LEN)
                     for r in subset]
            refs = [r["tgt"] for r in subset]
            score = metrics.score(preds, refs, f"nllb_{run_name}_{test_name}_{direction}")
            score.update({"run": run_name, "test_set": test_name, "direction": direction,
                         "n": len(subset)})
            results.append(score)

    del model
    torch.cuda.empty_cache()

import pandas as pd
results_df = pd.DataFrame(results)[["run", "test_set", "direction", "n", "chrf2pp", "bleu"]]
results_df.to_csv(f"{LOGS_DIR}/nllb_all_results.csv", index=False)
print(results_df.to_string(index=False))

# chrF2++ is the primary metric for this project (see metrics.py) - Ekegusii's
# agglutinative morphology penalizes BLEU's exact whole-word matching even for
# correct translations. Pivot to one row per run, one column per test set - averaged
# across en/sw directions - for an at-a-glance comparison across the three domains.
print("\nchrF2++ summary (mean across en/sw directions), per run x test set:")
summary = results_df.groupby(["run", "test_set"])["chrf2pp"].mean().unstack("test_set")
summary = summary[["bible", "psa", "general"]]
print(summary.round(1).to_string())
summary.to_csv(f"{LOGS_DIR}/nllb_chrf_summary.csv")

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_zeroshot_bible_en-guz          chrF2++=15.17  BLEU=0.09
nllb_zeroshot_bible_sw-guz          chrF2++=14.61  BLEU=0.64
nllb_zeroshot_psa_en-guz            chrF2++=14.95  BLEU=1.74
nllb_zeroshot_psa_sw-guz            chrF2++=14.76  BLEU=1.71
nllb_zeroshot_general_en-guz        chrF2++=16.91  BLEU=1.38
nllb_zeroshot_general_sw-guz        chrF2++=17.33  BLEU=1.43


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_stage1_lora_emb_bible_en-guz   chrF2++=41.53  BLEU=13.33
nllb_stage1_lora_emb_bible_sw-guz   chrF2++=39.77  BLEU=12.56
nllb_stage1_lora_emb_psa_en-guz     chrF2++=20.10  BLEU=1.94
nllb_stage1_lora_emb_psa_sw-guz     chrF2++=20.13  BLEU=1.76
nllb_stage1_lora_emb_general_en-guz chrF2++=33.16  BLEU=8.77
nllb_stage1_lora_emb_general_sw-guz chrF2++=26.54  BLEU=5.02


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

...

nllb_mixed_lora_emb_bible_en-guz    chrF2++=42.47  BLEU=14.06
nllb_mixed_lora_emb_bible_sw-guz    chrF2++=40.41  BLEU=13.10
nllb_mixed_lora_emb_psa_en-guz      chrF2++=28.71  BLEU=4.67
nllb_mixed_lora_emb_psa_sw-guz      chrF2++=27.11  BLEU=3.91
nllb_mixed_lora_emb_general_en-guz  chrF2++=41.00  BLEU=13.98
nllb_mixed_lora_emb_general_sw-guz  chrF2++=39.57  BLEU=13.98


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_psaonly_lora_only_bible_en-guz chrF2++=24.35  BLEU=1.63
nllb_psaonly_lora_only_bible_sw-guz chrF2++=23.42  BLEU=2.27
nllb_psaonly_lora_only_psa_en-guz   chrF2++=43.60  BLEU=19.58
nllb_psaonly_lora_only_psa_sw-guz   chrF2++=41.74  BLEU=18.35
nllb_psaonly_lora_only_general_en-guz chrF2++=36.56  BLEU=16.45
nllb_psaonly_lora_only_general_sw-guz chrF2++=45.67  BLEU=26.70


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_psaonly_lora_emb_bible_en-guz  chrF2++=18.03  BLEU=0.64
nllb_psaonly_lora_emb_bible_sw-guz  chrF2++=18.02  BLEU=1.10
nllb_psaonly_lora_emb_psa_en-guz    chrF2++=26.26  BLEU=3.81
nllb_psaonly_lora_emb_psa_sw-guz    chrF2++=25.15  BLEU=2.89
nllb_psaonly_lora_emb_general_en-guz chrF2++=26.20  BLEU=5.77
nllb_psaonly_lora_emb_general_sw-guz chrF2++=31.68  BLEU=9.20
              run test_set direction    n   chrf2pp      bleu
         zeroshot    bible    en-guz 1994 15.173291  0.094109
         zeroshot    bible    sw-guz  582 14.614486  0.637023
         zeroshot      psa    en-guz 1862 14.948234  1.741395
         zeroshot      psa    sw-guz 1158 14.757830  1.710378
         zeroshot  general    en-guz 5159 16.911124  1.378121
         zeroshot  general    sw-guz 2874 17.330070  1.431738
  stage1_lora_emb    bible    en-guz 1994 41.528237 13.331411
  stage1_lora_emb    bible    sw-guz  582 39.773252 12.563867
  stage1_lora_emb      psa    en-guz 1862 20.099756  1.938992
  stage1_lor

## 8. Save deployment-ready checkpoints

Bundles each merged checkpoint into `deployment/nllb/{run_id}/`, verifying a
`.safetensors` file is actually present - this is what the application loads from.

In [ ]:
for run_name, path in all_paths.items():
    inference.save_for_deployment(path, ARCHITECTURE, f"nllb_{run_name}", deployment_dir=DEPLOYMENT_DIR)


deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_zeroshot  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_stage1_lora_emb  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_stage1_lora_only  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_stage2_lora_only  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_stage2_lora_emb  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_mixed_lora_only  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_mixed_lora_emb  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_psaonly_lora_only  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_psaonly_lora_emb  (1 safetensors file

## 9. Demonstration

Pick whichever configuration Section 7's results say performed best, load it, and
translate a few real PSA-style sentences - this is the working translation demo.

In [32]:
import importlib
import train, inference
importlib.reload(train)
importlib.reload(inference)

fine_tuned_summary = summary.drop(index="zeroshot", errors="ignore")
avg_chrf2pp = fine_tuned_summary.mean(axis=1)
DEMO_RUN = avg_chrf2pp.idxmax()
print(f"selected: {DEMO_RUN}")

demo_path = all_paths[DEMO_RUN]
demo_model = AutoModelForSeq2SeqLM.from_pretrained(demo_path)
demo_model.to("cuda" if torch.cuda.is_available() else "cpu")
demo_tokenizer = AutoTokenizer.from_pretrained(demo_path)

samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),   # PSA register
    ("Ministry of Health: report suspected cholera cases immediately.", "en"),         # PSA register
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),  # PSA register
    ("In the beginning, God created the heavens and the earth.", "en"),               # scripture register
    ("The children are from school.", "en"),                                          # everyday/dictionary register
]

print(f"\nNLLB demo - configuration: {DEMO_RUN}\n")
for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  ->", inference.generate_one(demo_model, demo_tokenizer, text, lang, cfg, ARCHITECTURE, MAX_LEN))
    print()

selected: mixed_lora_only


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]


NLLB demo - configuration: mixed_lora_only

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Abaremi basoroirwe korwa ase ogotumia chingencho chioboremi bokebwenerete bosio bwa amagoro aa.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Ewizara ya obochenu: karwe amang'ana igoro y'oborwaria bwa cholela kegima.

[sw] Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.
  -> Abaremi baariganiigwe ase kobeka chindagera chibogisaria oboremi obuya amagoro aa.

[en] In the beginning, God created the heavens and the earth.
  -> Agwo omochakano Nyasae agatonga igoro na ense.

[en] The children are from school.
  -> Abana tibakori esukuru.

